In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

from metrics import (
    F1_score,
    IoU,
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
gt_videos_path = Path(
    "../benchmark/queries_and_videos_test.json"
)

all_possible_videos = list([f.stem for f in Path(
    "/media/EVO870/datasets/prompting-mammalps-v2/annotations/test"
).rglob("*.json")])

retrieved_results_path = Path(
    "/media/eceo_scratch_haas001/results/prompting_mammalps-v2/LLM/qwen/Qwen3-8B-salma.json"
)  # Path to the JSON file create by run.py

with open(gt_videos_path, "r") as f:
    gt_videos_cat = json.load(f)

gt_videos = {}
for cat in gt_videos_cat:
    
    for query, video_list  in gt_videos_cat[cat].items():
        gt_videos[query] = video_list

with open(retrieved_results_path, "r") as f:
    pred_videos = json.load(f)


with open("../benchmark/query_categories.json", "r") as f:
    query_categories = json.load(f)

query2eco_cat = {q: [] for q in gt_videos.keys()}
for cat in query_categories["ECOLOGY"]:
    for q in query_categories["ECOLOGY"][cat]:
        query2eco_cat[q].append(cat)

query2cv_cat = {q: [] for q in gt_videos.keys()}
for cat in query_categories["VISION"]:
    for q in query_categories["VISION"][cat]:
        query2cv_cat[q].append(cat)

# Overall performance

In [55]:
mF1, F1_queries = F1_score(gt_queries_videos=gt_videos, pred_queries_videos=pred_videos, all_videos=all_possible_videos)
mIoU, IoU_queries = IoU(gt_queries_videos=gt_videos, pred_queries_videos=pred_videos, all_videos=all_possible_videos)

In [56]:
print(f"F1-score (macro-avg.): {mF1:.2f}")
print(f"mean IoU: {mIoU:.2f}")

F1-score (macro-avg.): 0.28
mean IoU: 0.21


In [57]:
print(f"F1-score (macro-avg.) w/o Ref prompts: {np.nanmean([s for q, s in F1_queries.items() if '<vid>' not in q]):.2f}")
print(f"mean IoU w/o Ref prompts : {np.nanmean([s for q, s in IoU_queries.items() if '<vid>' not in q]):.2f}")

F1-score (macro-avg.) w/o Ref prompts: 0.30
mean IoU w/o Ref prompts : 0.23


# Performance per query et per category

In [32]:
results_df = pd.concat([
    pd.DataFrame.from_dict(IoU_queries, orient="index", columns=["mIoU"]),
    pd.DataFrame.from_dict(F1_queries, orient="index", columns=["F1-score"]),
    ], axis=1)

pd.set_option('display.max_rows', 150)
pd.set_option('display.max_colwidth', 90)
results_df.sort_values("mIoU", ascending=False)

,mIoU,F1-score
An adult female red deer escaping in sunny weather.,1.000000,1.000000
An adult male red deer doing the same sequence of actions while wallowing as any individual of <vid>S3_C2_E769_V0436</vid>.,1.000000,1.000000
A fox reacting to a camera.,1.000000,1.000000
A juvenile roe deer suckling.,1.000000,1.000000
A juvenile roe deer preparing to suckle.,1.000000,1.000000
A fox sniffing.,1.000000,1.000000
A juvenile roe deer.,0.998710,0.999354
A roe deer participating in courtship in rainy weather.,0.997419,0.998708
A fox chasing prey in clear weather.,0.997419,0.998708
A fox chasing prey.,0.997419,0.998708


In [33]:
eco_cat_df = pd.DataFrame([query2eco_cat], index=["eco_cat"]).T
cv_cat_df = pd.DataFrame([query2cv_cat], index=["cv_cat"]).T
results_cat_df = results_df.merge(eco_cat_df, left_index=True, right_index=True).merge(cv_cat_df, left_index=True, right_index=True)

In [34]:
# Ecology categories
results_cat_df[["mIoU", "F1-score", "eco_cat"]].explode(["eco_cat"]).groupby("eco_cat").mean().T

eco_cat,CAMERA_REACTION,COMMON,COURTSHIP,RARE,SOCIAL
mIoU,0.149804,0.630031,0.260414,0.337257,0.273017
F1-score,0.196993,0.772803,0.327327,0.425771,0.326541


In [35]:
results_cat_df[["mIoU", "F1-score", "eco_cat"]].explode(["eco_cat"]).groupby("eco_cat").count().T

eco_cat,CAMERA_REACTION,COMMON,COURTSHIP,RARE,SOCIAL
mIoU,28,2,27,48,36
F1-score,28,2,27,48,36


In [36]:
results_cat_df[["mIoU", "F1-score", "cv_cat"]].explode(["cv_cat"]).groupby("cv_cat").mean().T

cv_cat,COMPLEX,MULTI_ATTR,MULTI_INDIV,SINGLE_ATTR,VIDEO_COMPARISON
mIoU,0.173321,0.283342,0.157816,0.282278,0.174684
F1-score,0.238726,0.346988,0.210755,0.391673,0.236437


In [37]:
results_cat_df[["mIoU", "F1-score", "cv_cat"]].explode(["cv_cat"]).groupby("cv_cat").count().T

cv_cat,COMPLEX,MULTI_ATTR,MULTI_INDIV,SINGLE_ATTR,VIDEO_COMPARISON
mIoU,15,92,25,14,22
F1-score,15,92,25,14,22
